# SetSawa inspection

Read-only checks of the released snapshot. Extraction accuracy does not verify source claims or personal identity. Run from the repository root or notebooks folder.


In [ ]:
import json
from pathlib import Path
from collections import Counter
root = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
load = lambda name: json.loads((root / 'data/processed/clean' / (name + '.json')).read_text(encoding='utf-8'))
records, provenance, provinces = load('records'), load('provenance'), load('provinces')
assert (len(records), len(provenance), len(provinces)) == (3963, 3966, 77)
{'records': len(records), 'source_rows': len(provenance), 'provinces': len(provinces)}


## Duplicate accounting

The three additional occurrences remain in provenance. Do not deduplicate by masked identifier.


In [ ]:
links = Counter(row['record_id'] for row in provenance)
assert sum(n - 1 for n in links.values()) == 3
assert len(load('duplicates')) == 3
assert sum(r['score_adjustment_reported'] for r in records) == 148888
assert sum(p['source_score_sum'] for p in provinces) == 148954
{'duplicate_extra_occurrences': sum(n - 1 for n in links.values()), 'distinct_masked_ids': len({r['masked_id'] for r in records})}


## Geography and source coverage

These are source labels, not official entity IDs or population-adjusted rates.


In [ ]:
assert len(load('districts')) == 659
assert len(load('agency_locations')) == 2030
assert sum(p['record_count'] for p in provinces) == len(records)
[p['province_name_th'] for p in provinces if p['record_count'] == 0]


## Source locations

Inspect page references without printing personal fields by default.


In [ ]:
rid = records[0]['record_id']
[{k: p[k] for k in ('document_id', 'page_number', 'data_row_on_page', 'source_ordinal')} for p in provenance if p['record_id'] == rid]
